In [2]:
! pip install seaborn altair


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.2/731.2 kB 13.6 MB/s eta 0:00:0000:01


In [1]:
# Check where the computation is actually running
print("=== CHECKING COMPUTATION LOCATION ===")

import platform
import socket
import psutil
import os

# Check system information
print(f"Hostname: {socket.gethostname()}")
print(f"Platform: {platform.platform()}")
print(f"Machine: {platform.machine()}")
print(f"Processor: {platform.processor()}")

# Check if we're in Azure
print(f"\nEnvironment indicators:")
print(f"  Current user: {os.getenv('USER', 'unknown')}")
print(f"  Home directory: {os.getenv('HOME', 'unknown')}")
print(f"  Working directory: {os.getcwd()}")

# Check compute resources
print(f"\nCompute resources:")
print(f"  CPU cores: {psutil.cpu_count()}")
print(f"  Total RAM: {psutil.virtual_memory().total / (1024**3):.1f} GB")
print(f"  Available RAM: {psutil.virtual_memory().available / (1024**3):.1f} GB")

# Check if this looks like Azure ML compute
if 'azureuser' in os.getcwd() or 'batch/tasks' in os.getcwd():
    print(f"\nCONFIRMED: Running on Azure ML Compute Instance")
    print(f"  Evidence: Path contains Azure-specific directories")
else:
    print(f"\nWARNING: May be running locally")

# Check network interface (Azure VMs have specific patterns)
import subprocess
try:
    result = subprocess.run(['hostname', '-I'], capture_output=True, text=True)
    ip_addresses = result.stdout.strip()
    print(f"  IP addresses: {ip_addresses}")
    
    if '10.' in ip_addresses or '172.' in ip_addresses:
        print(f"  CONFIRMED: Private IP suggests cloud/Azure environment")
except:
    print(f"  Could not check IP addresses")

=== CHECKING COMPUTATION LOCATION ===
Hostname: pritshahlecha1
Platform: Linux-6.8.0-1026-azure-x86_64-with-glibc2.35
Machine: x86_64
Processor: x86_64

Environment indicators:
  Current user: azureuser
  Home directory: /home/azureuser
  Working directory: /mnt/batch/tasks/shared/LS_root/mounts/clusters/pritshahlecha1/code

Compute resources:
  CPU cores: 4
  Total RAM: 31.3 GB
  Available RAM: 28.2 GB

CONFIRMED: Running on Azure ML Compute Instance
  Evidence: Path contains Azure-specific directories
  IP addresses: 10.0.0.4 172.17.0.1
  CONFIRMED: Private IP suggests cloud/Azure environment


In [2]:
# Patient Readmission EDA with Polars
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import os
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv
from io import StringIO
print("✅ All libraries imported successfully!")
print(f"Polars version: {pl.__version__}")

✅ All libraries imported successfully!
Polars version: 1.30.0


In [3]:
# Navigate to the correct directory
import os
os.chdir('/mnt/batch/tasks/shared/LS_root/mounts/clusters/pritshahlecha1/code/patient-readmission-mlops')

print(f"Changed to: {os.getcwd()}")

# Check if .env file exists now
if os.path.exists('.env'):
    print("✅ .env file found!")
else:
    print("❌ .env file still not found")
    print("Available files:")
    for f in os.listdir('.'):
        if f.startswith('.env'):
            print(f"  {f}")

# Now try loading the environment
from dotenv import load_dotenv
load_dotenv()

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
print(f"Connection string loaded: {connection_string is not None}")

Changed to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/pritshahlecha1/code/patient-readmission-mlops
✅ .env file found!
Connection string loaded: True


In [4]:
# Load diabetes dataset from Azure Blob Storage
def load_diabetes_data_polars():
    """Load diabetes dataset using Polars"""
    load_dotenv()
    
    connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    container_name = "patient-data"
    
    # Get data from Azure Blob
    blob_service_client = BlobServiceClient.from_connection_string(connection_string)
    blob_client = blob_service_client.get_blob_client(
        container=container_name, 
        blob="raw-data/diabetic_data.csv"
    )
    
    # Download and read with Polars
    blob_data = blob_client.download_blob().readall()
    csv_string = blob_data.decode('utf-8')
    
    # Read with Polars
    df = pl.read_csv(StringIO(csv_string))
    
    return df

# Load the dataset
df = load_diabetes_data_polars()

print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {len(df.columns)}")

Dataset loaded successfully!
Shape: (101766, 50)
Columns: 50


In [5]:
# Basic dataset info
print("=== DATASET OVERVIEW ===")
print(f"Rows: {df.height:,}")
print(f"Columns: {df.width}")
print(f"Memory usage: {df.estimated_size('mb'):.2f} MB")

print("\n=== COLUMN TYPES ===")
print(df.dtypes)

print("\n=== FIRST 5 ROWS ===")
df.head()

=== DATASET OVERVIEW ===
Rows: 101,766
Columns: 50
Memory usage: 20.54 MB

=== COLUMN TYPES ===
[Int64, Int64, String, String, String, String, Int64, Int64, Int64, Int64, String, String, Int64, Int64, Int64, Int64, Int64, Int64, String, String, String, Int64, String, String, String, String, String, String, String, String, String, String, String, String, String, String, String, String, String, String, String, String, String, String, String, String, String, String, String, String]

=== FIRST 5 ROWS ===


encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
i64,i64,str,str,str,str,i64,i64,i64,i64,str,str,i64,i64,i64,i64,i64,i64,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
2278392,8222157,"""Caucasian""","""Female""","""[0-10)""","""?""",6,25,1,1,"""?""","""Pediatrics-Endocrinology""",41,0,1,0,0,0,"""250.83""","""?""","""?""",1,"""None""","""None""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""NO"""
149190,55629189,"""Caucasian""","""Female""","""[10-20)""","""?""",1,1,7,3,"""?""","""?""",59,0,18,0,0,0,"""276""","""250.01""","""255""",9,"""None""","""None""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Up""","""No""","""No""","""No""","""No""","""No""","""Ch""","""Yes""",""">30"""
64410,86047875,"""AfricanAmerican""","""Female""","""[20-30)""","""?""",1,1,7,2,"""?""","""?""",11,5,13,2,0,1,"""648""","""250""","""V27""",6,"""None""","""None""","""No""","""No""","""No""","""No""","""No""","""No""","""Steady""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""NO"""
500364,82442376,"""Caucasian""","""Male""","""[30-40)""","""?""",1,1,7,2,"""?""","""?""",44,1,16,0,0,0,"""8""","""250.43""","""403""",7,"""None""","""None""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Up""","""No""","""No""","""No""","""No""","""No""","""Ch""","""Yes""","""NO"""
16680,42519267,"""Caucasian""","""Male""","""[40-50)""","""?""",1,1,7,1,"""?""","""?""",51,0,8,0,0,0,"""197""","""157""","""250""",5,"""None""","""None""","""No""","""No""","""No""","""No""","""No""","""No""","""Steady""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Steady""","""No""","""No""","""No""","""No""","""No""","""Ch""","""Yes""","""NO"""


In [6]:
# STEP 1: Missing Value Detection
def detect_all_missing_patterns(df):
    """ missing value detection"""
    
    missing_report = []
    
    for col in df.columns:
        col_data = df[col]
        total_rows = df.height
        
        # Standard null detection
        null_count = col_data.null_count()
        
        # Healthcare-specific missing patterns
        if col_data.dtype == pl.Utf8:  # String columns
            question_marks = df.filter(pl.col(col) == "?").height
            empty_strings = df.filter(pl.col(col) == "").height
            na_values = df.filter(pl.col(col).str.to_lowercase().is_in(["na", "n/a", "null", "none", "unknown"])).height # we can add more
            missing_total = null_count + question_marks + empty_strings + na_values
        else:  # Numeric columns
            missing_total = null_count
            question_marks = empty_strings = na_values = 0
        
        missing_percentage = (missing_total / total_rows) * 100
        
        missing_report.append({
            'column': col,
            'total_missing': missing_total,
            'missing_percentage': round(missing_percentage, 2),
            'null_count': null_count,
            'question_marks': question_marks,
            'empty_strings': empty_strings,
            'na_values': na_values,
            'dtype': str(col_data.dtype)
        })
    
    return pl.DataFrame(missing_report)

# Run missing data detection
missing_report = detect_all_missing_patterns(df)
print("=== MISSING DATA ANALYSIS ===")
print(missing_report.filter(pl.col('total_missing') > 0).sort('missing_percentage', descending=True))

=== MISSING DATA ANALYSIS ===
shape: (9, 8)
┌────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬───────────┬────────┐
│ column     ┆ total_miss ┆ missing_pe ┆ null_count ┆ question_m ┆ empty_stri ┆ na_values ┆ dtype  │
│ ---        ┆ ing        ┆ rcentage   ┆ ---        ┆ arks       ┆ ngs        ┆ ---       ┆ ---    │
│ str        ┆ ---        ┆ ---        ┆ i64        ┆ ---        ┆ ---        ┆ i64       ┆ str    │
│            ┆ i64        ┆ f64        ┆            ┆ i64        ┆ i64        ┆           ┆        │
╞════════════╪════════════╪════════════╪════════════╪════════════╪════════════╪═══════════╪════════╡
│ weight     ┆ 98569      ┆ 96.86      ┆ 0          ┆ 98569      ┆ 0          ┆ 0         ┆ String │
│ max_glu_se ┆ 96420      ┆ 94.75      ┆ 0          ┆ 0          ┆ 0          ┆ 96420     ┆ String │
│ rum        ┆            ┆            ┆            ┆            ┆            ┆           ┆        │
│ A1Cresult  ┆ 84748      ┆ 83.28      ┆ 0     

In [7]:
# Missing data handling decisions
print("=== MISSING DATA DECISIONS ===")
print()

# Drop columns with >95% missing 
high_missing = missing_report.filter(pl.col('missing_percentage') > 95)
print("COLUMNS TO DROP (>95% missing):")
for row in high_missing.iter_rows(named=True):
   print(f"  {row['column']}: {row['missing_percentage']}% missing")

print()

# Features with 50-95% missing need careful consideration
medium_missing = missing_report.filter(
   (pl.col('missing_percentage') > 50) & (pl.col('missing_percentage') <= 95)
)
print("COLUMNS NEEDING DECISION (50-95% missing):")
for row in medium_missing.iter_rows(named=True):
   print(f"  {row['column']}: {row['missing_percentage']}% missing")

print()

# Low missing can usually be handled with imputation
low_missing = missing_report.filter(pl.col('missing_percentage') <= 50)
print("COLUMNS TO HANDLE (<50% missing):")
for row in low_missing.iter_rows(named=True):
   if row['total_missing'] > 0:
       print(f"  {row['column']}: {row['missing_percentage']}% missing")

=== MISSING DATA DECISIONS ===

COLUMNS TO DROP (>95% missing):
  weight: 96.86% missing

COLUMNS NEEDING DECISION (50-95% missing):
  max_glu_serum: 94.75% missing
  A1Cresult: 83.28% missing

COLUMNS TO HANDLE (<50% missing):
  race: 2.23% missing
  payer_code: 39.56% missing
  medical_specialty: 49.08% missing
  diag_1: 0.02% missing
  diag_2: 0.35% missing
  diag_3: 1.4% missing


In [8]:
# Step 1: Drop weight column
df_clean = df.drop('weight')
print("Dropped weight column (96.86% missing)")

# Check what the actual values are in A1Cresult
print("A1Cresult unique values:")
print(df_clean['A1Cresult'].unique().sort())

Dropped weight column (96.86% missing)
A1Cresult unique values:
shape: (4,)
Series: 'A1Cresult' [str]
[
	">7"
	">8"
	"None"
	"Norm"
]


In [9]:
# Import required libraries for KNN imputation
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Step 1: Prepare data for KNN imputation
print("=== PREPARING DATA FOR KNN IMPUTATION ===")

# Create a copy for KNN processing (Polars syntax)
df_knn = df_clean.clone()
print(f"Starting with shape: {df_knn.shape}")

# Check current missing patterns
print("\nCurrent missing value patterns:")
for col in ['race', 'medical_specialty', 'payer_code', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult']:
    if col in df_knn.columns:
        missing_count = df_knn.filter(pl.col(col).is_in(['?', 'None', ''])).height
        total_count = df_knn.height
        print(f"{col}: {missing_count} missing ({missing_count/total_count*100:.1f}%)")

=== PREPARING DATA FOR KNN IMPUTATION ===
Starting with shape: (101766, 49)

Current missing value patterns:
race: 2273 missing (2.2%)
medical_specialty: 49949 missing (49.1%)
payer_code: 40256 missing (39.6%)
diag_1: 21 missing (0.0%)
diag_2: 358 missing (0.4%)
diag_3: 1423 missing (1.4%)
max_glu_serum: 96420 missing (94.7%)
A1Cresult: 84748 missing (83.3%)


In [10]:
df_for_knn = df_knn.drop(['max_glu_serum', 'A1Cresult'])
print(f"Dataset shape after dropping high missing columns: {df_for_knn.shape}")

# Check remaining missing patterns
print("\n=== REMAINING MISSING PATTERNS ===")
remaining_missing_cols = ['race', 'medical_specialty', 'payer_code', 'diag_1', 'diag_2', 'diag_3']

for col in remaining_missing_cols:
    if col in df_for_knn.columns:
        missing_count = df_for_knn.filter(pl.col(col).is_in(['?', 'None', ''])).height
        total_count = df_for_knn.height
        print(f"{col}: {missing_count} missing ({missing_count/total_count*100:.1f}%)")

Dataset shape after dropping high missing columns: (101766, 47)

=== REMAINING MISSING PATTERNS ===
race: 2273 missing (2.2%)
medical_specialty: 49949 missing (49.1%)
payer_code: 40256 missing (39.6%)
diag_1: 21 missing (0.0%)
diag_2: 358 missing (0.4%)
diag_3: 1423 missing (1.4%)


In [11]:
# Step 3: Encode categorical variables for KNN imputation (Fixed)
print("=== ENCODING CATEGORICAL VARIABLES FOR KNN ===")

# Check data types first
print("Column data types:")
for col in df_for_knn.columns[:10]:  # Show first 10
    dtype = df_for_knn[col].dtype
    print(f"  {col}: {dtype}")

# Define columns to encode (only string columns need encoding)
string_cols = []
for col in df_for_knn.columns:
    if df_for_knn[col].dtype == pl.Utf8:  # String columns
        string_cols.append(col)

print(f"\nString columns to encode: {string_cols}")

# Start encoding process
df_encoded = df_for_knn.clone()

# Replace missing indicators with null (only for string columns)
for col in string_cols:
    df_encoded = df_encoded.with_columns(
        pl.when(pl.col(col).is_in(['?', 'None', ''])).then(None).otherwise(pl.col(col)).alias(col)
    )

print("Replaced missing indicators with null values in string columns")

# Continue with encoding
label_mappings = {}

for col in string_cols:
    # Get unique non-null values
    unique_values = df_encoded.filter(pl.col(col).is_not_null()).select(col).unique().sort(col)
    
    if unique_values.height > 0:
        # Create mapping dictionary
        unique_list = unique_values[col].to_list()
        mapping = {val: idx for idx, val in enumerate(unique_list)}
        label_mappings[col] = mapping
        
        # Apply encoding
        df_encoded = df_encoded.with_columns(
            pl.col(col).map_elements(
                lambda x: mapping.get(x, None) if x is not None else None, 
                return_dtype=pl.Float64
            ).alias(col)
        )
        
        print(f"  Encoded {col}: {len(mapping)} unique values")

print(f"\nEncoding complete! Dataset ready for KNN: {df_encoded.shape}")

=== ENCODING CATEGORICAL VARIABLES FOR KNN ===
Column data types:
  encounter_id: Int64
  patient_nbr: Int64
  race: String
  gender: String
  age: String
  admission_type_id: Int64
  discharge_disposition_id: Int64
  admission_source_id: Int64
  time_in_hospital: Int64
  payer_code: String

String columns to encode: ['race', 'gender', 'age', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']


Replaced missing indicators with null values in string columns
  Encoded race: 5 unique values
  Encoded gender: 3 unique values
  Encoded age: 10 unique values
  Encoded payer_code: 17 unique values
  Encoded medical_specialty: 72 unique values
  Encoded diag_1: 716 unique values
  Encoded diag_2: 748 unique values
  Encoded diag_3: 789 unique values
  Encoded metformin: 4 unique values
  Encoded repaglinide: 4 unique values
  Encoded nateglinide: 4 unique values
  Encoded chlorpropamide: 4 unique values
  Encoded glimepiride: 4 unique values
  Encoded acetohexamide: 2 unique values
  Encoded glipizide: 4 unique values
  Encoded glyburide: 4 unique values
  Encoded tolbutamide: 2 unique values
  Encoded pioglitazone: 4 unique values
  Encoded rosiglitazone: 4 unique values
  Encoded acarbose: 4 unique values
  Encoded miglitol: 4 unique values
  Encoded troglitazone: 2 unique values
  Encoded tolazamide: 3 unique values
  Encoded examide: 1 unique values
  Encoded citoglipton: 1 uniqu

In [12]:
import altair as alt
import pandas as pd
# Enable Altair to display in Jupyter
alt.data_transformers.enable('json')

print("=== EXPLORING DATA WITH ALTAIR VISUALIZATIONS ===")

# Convert small sample to pandas for Altair (Altair works best with pandas)
# Take a sample for plotting (Altair can be slow with 100K+ rows)
sample_size = 5000
df_sample = df_encoded.sample(sample_size, seed=42).to_pandas()

print(f"Created sample of {sample_size} patients for visualization")

# Visualization 1: Missing data patterns
print("\n1. Missing Data Patterns")
missing_data = []
for col in ['race', 'medical_specialty', 'payer_code', 'diag_1', 'diag_2', 'diag_3']:
    if col in df_sample.columns:
        missing_count = df_sample[col].isnull().sum()
        missing_pct = (missing_count / len(df_sample)) * 100
        missing_data.append({'column': col, 'missing_percentage': missing_pct})

missing_df = pd.DataFrame(missing_data)

chart1 = alt.Chart(missing_df).mark_bar().encode(
    x=alt.X('missing_percentage:Q', title='Missing Percentage'),
    y=alt.Y('column:N', title='Column', sort='-x'),
    color=alt.Color('missing_percentage:Q', scale=alt.Scale(scheme='reds'))
).properties(
    title='Missing Data Patterns Before KNN Imputation',
    width=400,
    height=300
)

chart1

=== EXPLORING DATA WITH ALTAIR VISUALIZATIONS ===
Created sample of 5000 patients for visualization

1. Missing Data Patterns


alt.Chart(...)

In [13]:
# Step 4: Apply KNN Imputation with Detailed Progress Monitoring
print("=== APPLYING KNN IMPUTATION WITH DETAILED MONITORING ===")

import time
import psutil
import os
from sklearn.impute import KNNImputer
import pandas as pd
import numpy as np

# Convert to pandas for KNN
df_pandas = df_encoded.to_pandas()
print(f"Converted to pandas: {df_pandas.shape}")

# Exclude ID columns from imputation
id_columns = ['encounter_id', 'patient_nbr']
columns_for_knn = [col for col in df_pandas.columns if col not in id_columns]

print(f"Applying KNN to {len(columns_for_knn)} columns (excluding ID columns)")

# Extract data for KNN
knn_data = df_pandas[columns_for_knn]

# Analyze missing patterns
missing_before = knn_data.isnull().sum().sum()
missing_by_column = knn_data.isnull().sum()
print(f"\nMissing value analysis:")
print(f"  Total missing values: {missing_before:,}")
print(f"  Percentage missing: {(missing_before / (knn_data.shape[0] * knn_data.shape[1])) * 100:.2f}%")
print(f"  Columns with missing data: {(missing_by_column > 0).sum()}")

# System resource monitoring
process = psutil.Process(os.getpid())
initial_memory = process.memory_info().rss / (1024 * 1024)  # MB
print(f"\nSystem resources at start:")
print(f"  Memory usage: {initial_memory:.1f} MB")
print(f"  CPU cores available: {psutil.cpu_count()}")

# Initialize KNN Imputer
knn_imputer = KNNImputer(n_neighbors=5)

print(f"\n{'='*60}")
print("KNN IMPUTATION PROCESS STARTING")
print(f"{'='*60}")
print(f"Algorithm: K-Nearest Neighbors Imputation")
print(f"K parameter: 5 neighbors")
print(f"Distance metric: Euclidean (default)")
print(f"Processing: {knn_data.shape[0]:,} samples x {knn_data.shape[1]} features")

# Calculate theoretical complexity
n_samples, n_features = knn_data.shape
complexity = n_samples * n_samples * n_features  # Approximate
print(f"Computational complexity: O(n²m) = {complexity:e} operations")

print(f"\nProcess stages:")
print(f"  1. Data validation and preprocessing")
print(f"  2. Distance matrix computation (most time-consuming)")
print(f"  3. Neighbor identification for each missing value")
print(f"  4. Imputation based on neighbor values")

# Start timing and monitoring
start_time = time.time()
print(f"\nStarting imputation at: {time.strftime('%H:%M:%S')}")

# Memory monitoring function
def check_memory():
    current_memory = process.memory_info().rss / (1024 * 1024)
    return current_memory

# Periodic monitoring during computation
import threading
import sys

class ProgressMonitor:
    def __init__(self):
        self.running = True
        self.start_memory = initial_memory
        
    def monitor(self):
        count = 0
        while self.running:
            time.sleep(30)  # Check every 30 seconds
            if self.running:
                count += 1
                elapsed = time.time() - start_time
                current_memory = check_memory()
                memory_increase = current_memory - self.start_memory
                
                print(f"\nProgress update {count}:")
                print(f"  Elapsed time: {elapsed/60:.1f} minutes")
                print(f"  Memory usage: {current_memory:.1f} MB (delta: +{memory_increase:.1f} MB)")
                print(f"  Process status: Computing distance matrices and imputing values")
                sys.stdout.flush()

# Start background monitoring
monitor = ProgressMonitor()
monitor_thread = threading.Thread(target=monitor.monitor)
monitor_thread.daemon = True
monitor_thread.start()

try:
    print("PHASE 1: Starting KNN fit_transform process...")
    
    # This is the main computation
    imputed_data = knn_imputer.fit_transform(knn_data)
    
    # Stop monitoring
    monitor.running = False
    
    # Calculate final stats
    end_time = time.time()
    duration = end_time - start_time
    final_memory = check_memory()
    memory_used = final_memory - initial_memory
    
    print(f"\nPHASE 2: Reconstruction and validation...")
    
    # Create DataFrame with imputed data
    df_imputed_pandas = pd.DataFrame(imputed_data, columns=columns_for_knn, index=df_pandas.index)
    
    # Add back ID columns
    df_final_pandas = pd.concat([df_pandas[id_columns], df_imputed_pandas], axis=1)
    
    # Convert back to Polars
    df_final = pl.from_pandas(df_final_pandas)
    
    print(f"PHASE 3: Quality assessment...")
    
    # Check for remaining missing values
    missing_count = df_final.null_count().sum_horizontal().sum()
    
    print(f"\n{'='*60}")
    print("KNN IMPUTATION COMPLETED SUCCESSFULLY")
    print(f"{'='*60}")
    print(f"Total processing time: {duration/60:.2f} minutes ({duration:.1f} seconds)")
    print(f"Processing rate: {knn_data.shape[0]/duration:.0f} samples/second")
    print(f"Memory peak usage: {final_memory:.1f} MB")
    print(f"Memory overhead: {memory_used:.1f} MB")
    print(f"Final dataset shape: {df_final.shape}")
    print(f"Missing values after imputation: {missing_count}")
    print(f"Imputation success rate: {((missing_before - missing_count) / missing_before * 100):.2f}%")
    
    if missing_count == 0:
        print("SUCCESS: All missing values have been successfully imputed")
    
except KeyboardInterrupt:
    monitor.running = False
    print(f"\nProcess interrupted by user after {(time.time() - start_time)/60:.1f} minutes")
except Exception as e:
    monitor.running = False
    print(f"\nError during KNN imputation: {str(e)}")
    print(f"Failed after {(time.time() - start_time)/60:.1f} minutes")

=== APPLYING KNN IMPUTATION WITH DETAILED MONITORING ===
Converted to pandas: (101766, 47)
Applying KNN to 45 columns (excluding ID columns)

Missing value analysis:
  Total missing values: 94,280
  Percentage missing: 2.06%
  Columns with missing data: 6

System resources at start:
  Memory usage: 844.8 MB
  CPU cores available: 4

KNN IMPUTATION PROCESS STARTING
Algorithm: K-Nearest Neighbors Imputation
K parameter: 5 neighbors
Distance metric: Euclidean (default)
Processing: 101,766 samples x 45 features
Computational complexity: O(n²m) = 4.660343e+11 operations

Process stages:
  1. Data validation and preprocessing
  2. Distance matrix computation (most time-consuming)
  3. Neighbor identification for each missing value
  4. Imputation based on neighbor values

Starting imputation at: 03:18:59
PHASE 1: Starting KNN fit_transform process...

Progress update 1:
  Elapsed time: 0.5 minutes
  Memory usage: 2933.6 MB (delta: +2088.8 MB)
  Process status: Computing distance matrices and

In [14]:
# Quick verification of our imputed dataset
print("=== POST-IMPUTATION DATASET VERIFICATION ===")

print(f"Dataset shape: {df_final.shape}")
print(f"Missing values: {df_final.null_count().sum_horizontal().sum()}")

# Check the target variable distribution  
print("\nTarget variable (readmitted) distribution:")
readmission_counts = df_final.group_by('readmitted').count().sort('readmitted')
print(readmission_counts)

# Sample of imputed data
print(f"\nFirst 5 rows of clean dataset:")
print(df_final.head())

print("\nReady for EDA and modeling!")

=== POST-IMPUTATION DATASET VERIFICATION ===
Dataset shape: (101766, 47)
Missing values: 0

Target variable (readmitted) distribution:
shape: (3, 2)
┌────────────┬───────┐
│ readmitted ┆ count │
│ ---        ┆ ---   │
│ f64        ┆ u32   │
╞════════════╪═══════╡
│ 0.0        ┆ 11357 │
│ 1.0        ┆ 35545 │
│ 2.0        ┆ 54864 │
└────────────┴───────┘

First 5 rows of clean dataset:
shape: (5, 47)
┌──────────────┬─────────────┬──────┬────────┬───┬─────────────┬────────┬─────────────┬────────────┐
│ encounter_id ┆ patient_nbr ┆ race ┆ gender ┆ … ┆ metformin-p ┆ change ┆ diabetesMed ┆ readmitted │
│ ---          ┆ ---         ┆ ---  ┆ ---    ┆   ┆ ioglitazone ┆ ---    ┆ ---         ┆ ---        │
│ i64          ┆ i64         ┆ f64  ┆ f64    ┆   ┆ ---         ┆ f64    ┆ f64         ┆ f64        │
│              ┆             ┆      ┆        ┆   ┆ f64         ┆        ┆             ┆            │
╞══════════════╪═════════════╪══════╪════════╪═══╪═════════════╪════════╪═════════════╪═════

/tmp/ipykernel_21481/1280843026.py:9: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  readmission_counts = df_final.group_by('readmitted').count().sort('readmitted')


In [15]:
# Decode the target variable and create binary classification
print("=== TARGET VARIABLE ANALYSIS ===")

# The readmitted values are encoded:
# Based on original research: 0=<30 days, 1=>30 days, 2=NO readmission
print("Readmitted variable encoding:")
print("  0.0 = Readmitted within 30 days (11,357 patients - 11.2%)")
print("  1.0 = Readmitted after 30 days (35,545 patients - 34.9%)")  
print("  2.0 = No readmission (54,864 patients - 53.9%)")

# Create binary target for 30-day readmission prediction (industry standard)
df_binary = df_final.with_columns([
    pl.when(pl.col('readmitted') == 0.0)
    .then(1)  # 30-day readmission = 1
    .otherwise(0)  # No 30-day readmission = 0
    .alias('readmitted_30_days')
])

# Check binary distribution
binary_counts = df_binary.group_by('readmitted_30_days').len().sort('readmitted_30_days')
print(f"\nBinary 30-day readmission target:")
print(binary_counts)

# Calculate percentages
total = df_binary.height
for row in binary_counts.iter_rows(named=True):
    pct = (row['len'] / total) * 100
    label = "30-day readmission" if row['readmitted_30_days'] == 1 else "No 30-day readmission"
    print(f"  {label}: {row['len']:,} ({pct:.1f}%)")

print(f"\nDataset ready for 30-day readmission prediction modeling!")
print(f"Final shape: {df_binary.shape}")

=== TARGET VARIABLE ANALYSIS ===
Readmitted variable encoding:
  0.0 = Readmitted within 30 days (11,357 patients - 11.2%)
  1.0 = Readmitted after 30 days (35,545 patients - 34.9%)
  2.0 = No readmission (54,864 patients - 53.9%)

Binary 30-day readmission target:
shape: (2, 2)
┌────────────────────┬───────┐
│ readmitted_30_days ┆ len   │
│ ---                ┆ ---   │
│ i32                ┆ u32   │
╞════════════════════╪═══════╡
│ 0                  ┆ 90409 │
│ 1                  ┆ 11357 │
└────────────────────┴───────┘
  No 30-day readmission: 90,409 (88.8%)
  30-day readmission: 11,357 (11.2%)

Dataset ready for 30-day readmission prediction modeling!
Final shape: (101766, 48)


In [18]:
# Save cleaned dataset to Azure Blob Storage
print("=== SAVING CLEANED DATASET TO AZURE BLOB STORAGE ===")

import os
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv
import tempfile

# Load environment
load_dotenv()
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
container_name = "patient-data"

# Connect to blob storage
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# Convert to CSV format for saving
df_to_save = df_binary.clone()

# Create a temporary file to save the CSV
with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False) as temp_file:
    # Write CSV to temporary file
    df_to_save.write_csv(temp_file.name)
    temp_filename = temp_file.name

print(f"Created temporary CSV file: {temp_filename}")

# Upload to blob storage in cleaned-data folder
blob_name = "cleaned-data/diabetic_data_cleaned_knn_imputed.csv"
blob_client = blob_service_client.get_blob_client(
    container=container_name, 
    blob=blob_name
)

# Upload the file
with open(temp_filename, 'rb') as data:
    blob_client.upload_blob(data, overwrite=True)

print(f"✅ Uploaded cleaned dataset to: {blob_name}")
print(f"📊 Dataset details:")
print(f"   Rows: {df_binary.shape[0]:,}")
print(f"   Columns: {df_binary.shape[1]}")
print(f"   Missing values: 0 (KNN imputed)")
print(f"   Target: readmitted_30_days (binary classification)")
print(f"   Features: All categorical variables encoded as numbers")

# Clean up temporary file
os.unlink(temp_filename)
print(f"Cleaned up temporary file")

# Also save metadata about the cleaning process
metadata = f"""
# Cleaned Diabetes Dataset Metadata

## Processing Steps Applied:
1. Dropped high missing columns: weight (96.86%), max_glu_serum (94.7%), A1Cresult (83.3%)
2. Applied KNN imputation (k=5) to remaining missing values
3. All categorical variables encoded as numerical values
4. Created binary target: readmitted_30_days (0=No, 1=Yes)

## Final Dataset:
- Shape: {df_binary.shape[0]:,} rows × {df_binary.shape[1]} columns
- Missing values: 0
- Target distribution: 11.2% readmitted within 30 days

## File Location: 
- Container: patient-data
- Path: cleaned-data/diabetic_data_cleaned_knn_imputed.csv

## Processing Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

# Save metadata
metadata_blob = "cleaned-data/dataset_metadata.txt"
metadata_client = blob_service_client.get_blob_client(container=container_name, blob=metadata_blob)
metadata_client.upload_blob(metadata.encode('utf-8'), overwrite=True)

print(f"✅ Saved processing metadata to: {metadata_blob}")
print(f"\n🎉 Cleaned dataset ready for future modeling and analysis!")

=== SAVING CLEANED DATASET TO AZURE BLOB STORAGE ===
Created temporary CSV file: /tmp/tmplrj444s3.csv
✅ Uploaded cleaned dataset to: cleaned-data/diabetic_data_cleaned_knn_imputed.csv
📊 Dataset details:
   Rows: 101,766
   Columns: 48
   Missing values: 0 (KNN imputed)
   Target: readmitted_30_days (binary classification)
   Features: All categorical variables encoded as numbers
Cleaned up temporary file
✅ Saved processing metadata to: cleaned-data/dataset_metadata.txt

🎉 Cleaned dataset ready for future modeling and analysis!
